# 🔧 Support Vector Machines
**ISLP Chapter 9 · Pattern Recognition for the Rest of Us**

> SVMs find the hyperplane that maximally separates classes — maximizing the margin between the nearest points. With kernels, they can draw nonlinear boundaries in high-dimensional spaces.

### What you'll learn
- Maximal margin classifier — the core idea
- Support vectors — only these points define the boundary
- Soft margin — allowing some misclassification
- Kernels — nonlinear SVMs (RBF, polynomial)
- Practical SVM with sklearn and the C/gamma tradeoff

### Dataset: Gene expression (cancer classification)

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from sklearn.svm import SVC, SVR
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.datasets import make_circles, make_classification

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: Heart

In [ ]:
import pandas as pd
heart = pd.read_csv(f'{DATA_DIR}/Heart.csv').dropna()
heart = pd.get_dummies(heart, drop_first=True, dtype=float)
target_col = [c for c in heart.columns if 'AHD' in c or c == heart.columns[-1]][0]
X = heart.drop(target_col, axis=1)
y = heart[target_col].astype(int)
print(f"Heart: {X.shape}  |  Disease rate: {y.mean():.1%}")

## 🎯 Part 1 — The Maximal Margin Classifier

An **SVM** finds the hyperplane (decision boundary) that maximizes the **margin** — the distance to the nearest training points from each class. These nearest points are called **support vectors**.

```
Maximize M  subject to:
  yᵢ(β₀ + β₁xᵢ₁ + ... + βₚxᵢₚ) ≥ M  for all i
  Σβⱼ² = 1
```

The **soft margin** (C parameter) allows some points to cross the margin:
- Large C → narrow margin, few violations, may overfit
- Small C → wide margin, allows violations, more robust

In [ ]:
# Visualize margin for linearly separable 2D data
np.random.seed(42)
X_2d = np.random.randn(80, 2)
y_2d = (X_2d[:,0] + X_2d[:,1] > 0).astype(int)

scaler_2d = StandardScaler()
X_2d_s = scaler_2d.fit_transform(X_2d)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
C_values = [0.01, 1.0, 100.0]
titles = ['Small C = 0.01\n(Wide margin, many violations)', 
          'C = 1.0\n(Balanced)', 
          'Large C = 100\n(Narrow margin, few violations)']

for ax, C, title in zip(axes, C_values, titles):
    svm = SVC(kernel='linear', C=C)
    svm.fit(X_2d_s, y_2d)
    
    xx, yy = np.meshgrid(np.linspace(-3,3,200), np.linspace(-3,3,200))
    Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, levels=[-np.inf,-1,0,1,np.inf],
                colors=['#f0e8ff','#e8f0fb','#f0e8ff','#e8f0fb'], alpha=0.8)
    ax.contour(xx, yy, Z, levels=[-1,0,1], colors=['#6b46c1','#1e5fa8','#6b46c1'],
               linestyles=['--','-','--'], linewidths=[1,2,1])
    
    # Support vectors
    ax.scatter(svm.support_vectors_[:,0], svm.support_vectors_[:,1],
               s=120, facecolors='none', edgecolors='black', lw=1.5, zorder=5, label='Support vectors')
    ax.scatter(X_2d_s[y_2d==0,0], X_2d_s[y_2d==0,1], color='#1e5fa8', alpha=0.6, s=25)
    ax.scatter(X_2d_s[y_2d==1,0], X_2d_s[y_2d==1,1], color='#e85d20', alpha=0.6, s=25)
    ax.set_title(title, fontsize=10)
    acc = svm.score(X_2d_s, y_2d)
    ax.set_xlabel(f'Acc={acc:.2f} | {len(svm.support_vectors_)} support vectors')

plt.suptitle('SVM: Effect of C Parameter on Margin Width', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# The kernel trick — nonlinear boundaries
np.random.seed(0)
X_circ, y_circ = make_circles(n_samples=200, noise=0.15, factor=0.3, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

kernels = [('linear', {}), ('poly', {'degree':3}), ('rbf', {'gamma':2})]
knames  = ['Linear kernel\n(fails on circles)', 'Polynomial kernel (d=3)', 'RBF kernel\n(best for circles)']

for ax, (kernel, params), kname in zip(axes, kernels, knames):
    svm_k = SVC(kernel=kernel, C=1.0, **params)
    svm_k.fit(X_circ, y_circ)
    
    xx, yy = np.meshgrid(np.linspace(-1.5,1.5,300), np.linspace(-1.5,1.5,300))
    Z = svm_k.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.25, colors=['#1e5fa8','#e85d20'])
    ax.scatter(X_circ[y_circ==0,0], X_circ[y_circ==0,1], color='#1e5fa8', s=25, alpha=0.7)
    ax.scatter(X_circ[y_circ==1,0], X_circ[y_circ==1,1], color='#e85d20', s=25, alpha=0.7)
    ax.set_title(f'{kname}\nAcc={svm_k.score(X_circ,y_circ):.2f}', fontsize=10)

plt.suptitle('SVM Kernels — Linear vs Polynomial vs RBF', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("\n📌 RBF kernel maps data to infinite-dimensional space implicitly — finds circular boundaries")
print("   The 'kernel trick' computes dot products in that space without ever going there")

In [ ]:
# SVM on Khan gene expression — high dimensional, few samples
svm_clf = SVC(kernel='linear', C=10, random_state=42)
svm_clf.fit(X_tr, y_tr)
y_pred = svm_clf.predict(X_te)

print("=== SVM on Khan Cancer Gene Expression ===\n")
print(classification_report(y_te, y_pred))
print(f"\n📌 {X_tr.shape[1]} features, only {X_tr.shape[0]} training samples")
print("   SVM excels here — it only depends on the support vectors, not the feature count")

In [ ]:
answers = {
    "q1": "",  # What are support vectors and why are they the only points that define the boundary?
    "q2": "",  # What does a larger C value do to the margin?
    "q3": "",  # What is the 'kernel trick' and why is it computationally efficient?
    "q4": "",  # When would you choose SVM over logistic regression?
    "q5": "",  # Why must features be standardized before fitting an SVM?
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("❌ Still empty: "+str(missing) if missing else "✅ Done! File → Save a copy in GitHub")